In [1]:
print("Hello World")

Hello World


In [2]:
import os
from dotenv import load_dotenv 
import tools as custom_tools
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, END, START

import operator
from typing import Annotated, TypedDict
load_dotenv()

True

In [3]:
calculator = custom_tools.calc
calculator.invoke({"expression": "2+2"})

'4'

In [4]:
llm = ChatOpenAI(
  base_url=os.getenv("TOGETHER_BASE_URL"),
  api_key=os.getenv("TOGETHER_API_KEY"),
  model=os.getenv("QWEN_MODEL")
)

all_tools =[custom_tools.calc, custom_tools.get_weather, custom_tools.web_search]

In [5]:
class AgentState(TypedDict):
  messages: Annotated[list, operator.add]

def agent_node(state:AgentState):
  messages = state['messages']

  llm_with_tools = llm.bind_tools(all_tools)
  resp = llm_with_tools.invoke(messages)
  if hasattr(resp, "tool_calls") and resp.tool_calls:
    for tc in resp.tool_calls:
      print(f"[AGENT] called Tool {tc.get("name", '?')} with args {tc.get('args', '?')}")
  else:
    print("[AGENT] Responding....")

  return {
    "messages": [resp]
  }


In [6]:
from langchain_core.messages import HumanMessage


state = {
  "messages": [HumanMessage("what is the latest news on HDFC?")]
}

resp = agent_node(state)


[AGENT] called Tool web_search with args {'query': 'latest news HDFC'}


In [7]:
resp

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 641, 'total_tokens': 733, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3.5-9B', 'system_fingerprint': None, 'id': 'onxNVSV-4YNCb4-a0ae1ebd2beb5b98', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ebf18-71ee-7d20-b133-f1bc4abb2a4e-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'latest news HDFC'}, 'id': 'call_6815ca421f754c8ba1a98ec2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 641, 'output_tokens': 92, 'total_tokens': 733, 'input_token_details': {}, 'output_token_details': {}})]}

In [8]:
def should_continue(state:AgentState):
  last_message = state['messages'][-1]
  if hasattr(last_message, "tool_calls") and last_message.tool_calls:
    return "tools"
  return END

In [9]:
def create_agent(checkpointer):
  builder = StateGraph(AgentState)
  builder.add_node("agent", agent_node)
  builder.add_node("tools", ToolNode(all_tools))

  builder.add_edge(START, "agent")
  builder.add_conditional_edges("agent", should_continue, ['tools', END])
  builder.add_edge("tools", "agent")
  graph = builder.compile(checkpointer=checkpointer)
  return graph

In [ ]:
def chat(agent, query, thread_id):

    config = {"configurable": {"thread_id": thread_id}}

    for chunk in agent.stream({'messages': [query]}, config=config):

        if 'agent' in chunk:
            chunk = chunk.get('agent')
        else:
            chunk = chunk.get('tools')

        if hasattr(chunk, 'tool_calls') and chunk.tool_calls:
            for tc in chunk.tool_calls:
                print(f"[AGENT] called Tool {tc.get('name', '?')} with args {tc.get('args', '?')}")
        else:
            print(f"[AGENT/ToolMessage] Responding.\n{chunk['messages'][0].content}")




## Postgres

In [28]:
from langgraph.checkpoint.postgres import PostgresSaver  
DB_URI = os.getenv("DATABASE_URL")


In [31]:
with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
  checkpointer.setup()
  agent = create_agent(checkpointer)
  chat(agent, "What is the weather in Hyderabad?", "demo")

[AGENT] called Tool weather with args {'location': 'Hyderabad'}
[AGENT/ToolMessage] Responding.

[AGENT/ToolMessage] Responding.
{"current_condition": [{"FeelsLikeC": "30", "FeelsLikeF": "86", "cloudcover": "75", "humidity": "74", "observation_time": "03:21 AM", "precipInches": "0.0", "precipMM": "0.0", "pressure": "1012", "pressureInches": "30", "temp_C": "27", "temp_F": "81", "uvIndex": "1", "visibility": "4", "visibilityMiles": "2", "weatherCode": "143", "weatherDesc": [{"value": "Mist"}], "weatherIconUrl": [{"value": "https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0006_mist.png"}], "winddir16Point": "W", "winddirDegree": "263", "windspeedKmph": "22", "windspeedMiles": "14"}], "nearest_area": [{"areaName": [{"value": "Hyderabad"}], "country": [{"value": "India"}], "latitude": "17.375", "longitude": "78.474", "population": "3597816", "region": [{"value": "Telangana"}], "weatherUrl": [{"value": "https://www.worldweatheronline.com/v2/weather.aspx?q=17.375,78.474"}]